In [46]:
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

In [47]:
#First we'll put in the numerical parameters that the Newman paper introduces.

cycleDemand = np.array([[35,450],[10,12],[35,-70],[10,12]]) #The first column is the length of each component of a single cycle (minutes), and the second column is the demand for that component (kW).

In [48]:
#Here we have the names of the parameters of the SEI model which Newman et al used.

#I'm guessing that this is far from a complete list of parameters I will likely use because of what GT-AutoLion used which wasn't published.
#I'll see what is possible to recreate some of their methods, unpublished parameters.

#This is also not including any state variables of the SEI model.

#Unless otherwise stated, the parameters here were gotten from either the Newman et al. paper or the data of their paraameters provided to GT-Autolion linked from the paper.

parameters_names = [["del_sei_0", "initial SEI thickness", "nm"], 
              ["kappa_sei,Tref", "Film conductivity at Tref", "S/m"],  #This one is referenced as a parameter in Newman et al., but no equation uses it; therefore I guess GT-AutoLion used it internally and I'll have to figure out how to implement it substantially similarly to how GT-AutoLion does it. It does get augmented with equation (1d) as a temperature-dependent term but that does not fully explain its role.
              ["E_a,kappa_film", "Film conductivity at activation energy", "J/mol"], #"
              ["D_EC,Tref", "EC diffusivity at Tref", "m2/s"], 
              ["E_a,D_EC", "EC diffusivity at activation energy", "J/mol"],
              ["k_0,sei,Tref", "SEI reaction rate constant at Tref", "-"], 
              ["E_a,k_0,sei", "SEI reaction rate activation energy", "J/mol"],  #I do not yet know why battery models use the notation "k_0" to denote the activation energy; is "k" reserved for something more generalized?
              ["U_sei", "SEI equilibrium potential", "V"],
              ["alpha_c,sei", "SEI charge transfer coefficient", "-"],
              ["M_sei", "SEI molecular weight", "g/mol"],
              ["rho_sei", "SEI density", "g/m3"],
              ["epsilon_sei", "SEI porosity", "-"],
              ["F", "Faraday constant", "C/mol"],
              ["n", "Bruggeman exponent", "-"],
              ["R", "Universal gas constant", "J/(mol*K)"],
              ["m_anode", "Anode mass fraction", "-"],
              ["rho_anode", "Anode density", "g/cm3"],
              ["R_a", "Anode particle radius", "micron"],
              ["m_binder", "Binder mass fraction", "-"],
              ["rho_binder", "Binder density", "g/cm3"],
              ["m_cond", "Conducting agent mass fraction", "-"],
              ["rho_cond", "Conducting agent density", "g/cm3"],
              ["phi_EC", "EC volume fraction", "-"],
              ["rho_elyte", "Electrolyte density", "g/cm3"],
              ["M_EC", "EC molecular weight", "g/mol"]
]

#This will be used to call parameters. The first value is the parameter for the NMC LI batteries, and the second is for the LFP LI batteries.

prms = { "del_sei_0": np.array([5,5]),
         "kappa_sei,Tref": np.array([1.5*10**-5, 1.5*10**-5]),
         "E_a,kappa_film": np.array([3.2*10**4, 3.2*10**4]),
         "D_EC,Tref": np.array([7*10**-19, 9.4*10**-21]),
         "E_a,D_EC": np.array([7*10**4, 7*10**4]),
         "k_0,sei,Tref": np.array([4.8*10**-15, 1.9*10**-17]),
         "E_a,k_0,sei": np.array([6*10**4, 6*10**4]),
         "U_sei": np.array([0.4, 0.4]),
         "alpha_c,sei": np.array([0.5, 0.5]),
         "M_sei": np.array([162, 162]),
         "rho_sei": np.array([1.69, 1.69]),
         "epsilon_sei": np.array([.05, .05]), 
         "F": np.array([96485, 96485]),
         "n": np.array([1.5, 1.5]),
         "R": np.array([8.314, 8.314]),
         "m_anode": np.array([0.94, 0.94]),
         "rho_anode": np.array([2.24, 2.24]),
         "R_a": np.array([12.53, 10.53]),
         "m_binder": np.array([0.03, 0.03]),
         "rho_binder": np.array([1.77, 1.77]),
         "m_cond": np.array([0.03, 0.03]),
         "rho_cond": np.array([1.95, 1.95]),
         "phi_EC": np.array([0.3, 0.3]),
         "rho_elyte": np.array([1.2, 1.2]),
         "M_EC": np.array([88.06, 88.06])
         }

#Double-checks that parameters_names and prms have the same length and consistent keys.
assert len(parameters_names) == len(prms), "Mismatch between number of parameter names and prms dictionary"
for name in parameters_names:
    assert name[0] in prms, f"Parameter {name[0]} is missing in prms dictionary"
assert all(key in [name[0] for name in parameters_names] for key in prms), "Mismatch between prms dictionary keys and parameter names"

#Just in case it's more practical to work with, we're going to also create a numpy array of the parameters.
prms_arr = np.array([prms[name[0]] for name in parameters_names])
prms_arr.shape

(25, 2)

In [49]:
#Great! Now for the state variables from Newman et al.:
#Note that the complete list of terms from table 2 is not here as some are redundant from the parameters table (table 1) hence included above.

variables_names = [
    ["a_s", "specific surface area for SEI reaction", "m2/m3"],
    ["c_EC", "EC concentration", "mol/m3"],
    ["C_EC,s", "EC concentration at the SEI surface", "mol/m2"],
    ["D_EC", "EC diffusivity", "m2/s"],
    ["D_EC,eff", "effective EC diffusivity", "m2/s"],
    ["i_0,sei", "SEI reaction exchange current density", "A/m2"],
    ["i_sei", "SEI reaction current density", "A/m2"],
    ["j_sei", "volumetric SEI reaction current density", "A/m3"],
    ["j_Li", "total current density", "A/m3"],
    ["k_0,sei", "SEI reaction rate constant", "1/s"],
    ["R_sei", "SEI layer resistance", "Ohm*m2"],
    ["T", "temperature", "K"],
    ["delta_sei", "SEI layer thickness", "m"],
    ["epsilon", "electrode porosity", "-"],
    ["kappa_sei", "SEI film conductivity", "S/m"],
    ["kappa_sei,eff", "effective SEI film conductivity", "S/m"],
    ["phi_e", "electrolyte potential", "V"],
    ["phi_s", "electrode potential", "V"]
]

#I guess that for now the state variables will be stored in a numpy vector and/or array. This dictionary will help us keep track of them by name by relating symbol and index.

variables = {
    "a_s": 0,
    "c_EC": 1,
    "C_EC,s": 2,
    "D_EC": 3,
    "D_EC,eff": 4,
    "i_0,sei": 5,
    "i_sei": 6,
    "j_sei": 7,
    "j_Li": 8,
    "k_0,sei": 9,
    "R_sei": 10,
    "T": 11,
    "delta_sei": 12,
    "epsilon": 13,
    "kappa_sei": 14,
    "kappa_sei,eff": 15,
    "phi_e": 16,
    "phi_s": 17
}

#Check that the variables dictionary and variables_names list have consistent keys and indices.
assert len(variables_names) == len(variables), "Mismatch between number of variable names and variables dictionary"
for name in variables_names:
    assert name[0] in variables, f"Variable {name[0]} is missing in variables dictionary"
assert all(key in [name[0] for name in variables_names] for key in variables), "Mismatch between variables dictionary keys and variable names"

In [50]:
#I guess that the next thing to do is to create a numpy array for the state variables similar to what we did for the parameters.
v = np.array([0 for _ in variables_names], dtype=float)
v.shape

(18,)

In [51]:
#At this point we need to decide which type of battery we want to try.
battery_type = "NMC"   #This accomodates NMC or LFP

if battery_type == "NMC":
    battery_indx = 0
elif battery_type == "LFP":
    battery_indx = 1
else:
    raise ValueError("Unsupported battery type")

In [52]:
#And now we do what we can to initialize the state variables that we know.

#This first one is for the specific surface area. We calculate it from the volume fraction (line 1) and the particle radius (line 2).
anode_volume_fraction = prms["m_anode"]/prms["rho_anode"] / (prms["m_anode"]/prms["rho_anode"] + prms["m_binder"]/prms["rho_binder"] + prms["m_cond"]/prms["rho_cond"])
v[variables["a_s"]] = (3 * anode_volume_fraction / (prms["R_a"]*10**-6))[battery_indx]

\begin{equation}
\varepsilon_s
=
\frac{\dfrac{m_s}{\rho_s}}
{\dfrac{m_s}{\rho_s}
+\dfrac{m_{\mathrm{bind}}}{\rho_{\mathrm{bind}}}
+\dfrac{m_{\mathrm{cond}}}{\rho_{\mathrm{cond}}}}
\end{equation}

\begin{equation}
a_s = \frac{3\varepsilon_s}{R_p},
\end{equation}

To effectively calculate initial EC concentration and concentration at the Electrode surface, we need to make some assumptions:

The density of the ethylene carbonate is equal to the density of the whole electrolyte solution
\begin{equation}
\rho_{EC}
\approx
\rho_{elyte}
\end{equation}

That allows for us to use the substitution in the following equation:

\begin{equation}
c_{EC,0} 
=
\frac
    {\phi_{EC} \rho_{EC}}
    {M_{EC}}

\end{equation}

Therefore,

\begin{equation}
c_{EC,0}
\approx
\frac
    {\varepsilon_{EC} \rho_{elyte}}
    {M_{EC}}
\end{equation}

The molar mass of Ethylene carbonate according to wikipedia is 88.06 g/mol.

In [53]:
v[variables["c_EC"]] = (prms["phi_EC"]*prms["rho_elyte"]/prms["M_EC"])[battery_indx]*(10**3)**3
v[variables["c_EC"]]

np.float64(4088121.735180558)

For the *surface* concentration in units of square meters, we need to establish an understanding of how thick we should consider the region from which EC is vulnerable to becoming SEI.

The Electrical Double Layer (EDL) is the thin, thin layer of electrolyte solution immediately in contact with the graphite which, upon polarization of the graphite, experiences the collection of positive ions attracted from the solution to the graphite's surphace.

This is where the electron field is the strongest, where Lithium, EC, and electrons are all together, and where electron transfer occurs. In essence, it is the place where our SEI reaction occurs.

If this were our reaction region, then its thickness (Ideally approximated to 1 nm) would be the conversion factor between bullk concentration at the surface and surface concentration:

\begin{equation}
c_{\text{EC}}^s
=
\delta_{\text{rxn}}c_{\text{EC}}
\end{equation}

where $\delta_{\text{EC}} = 10^{-9}$ as our units are in meters.

How can we be sure this is a reasonable interpretation?

Well, we can test to see if it's a reasonable ballpark number by remembering that Newman et al. also provide a value for the SEI reaction rate constant, $k_{0,\text{SEI}}$ on the order of $10^{-15}$ and $10^{-16}$ for NMC and LFP batteries, respectively.


Note that I'm going to defer to the values in the "simulation-parameters.xlsx" sheet where the values disagree, as I have above. 

However, note that for the NMC sheet, the unit for $k_{0,\text{SEI}}$ isn't listed, and for the LFP sheet, its unit is S/m. My best guess as that the Newman et al. intended to type m/s which is a plausible unit for  $k_{0,\text{SEI}}$ and mistakenly typed s/m

But we digress. In other literature we look to see if $k_{0,\text{SEI}}$ values on such an order is reasonable for surface concentrations yielded by our assumed reaction region thickness of 1 nm.

Ma et al. also have a reaction rate constant at $10^{-16}$-ish. They calculate $c_{\text{EC}}^s$ as a volumetric concentration implied in equation (19) from their paper:

\begin{equation}
-D_{\mathrm{EC}}
\frac{c_{\mathrm{EC}}^{s}-c_{\mathrm{EC}}^{0}}
{\delta_{\mathrm{film}}}
=
-\frac{j_{\mathrm{SEI}}}{F}
\end{equation}

which breaks down into the units:

\begin{equation}
\left[\frac{\mathrm{m}^2}{\mathrm{s}}\right]
\left[
\frac{\mathrm{mol}\,\mathrm{m}^{-3}}
{\mathrm{m}}
\right]
=
\left[
\frac{\mathrm{C}\,\mathrm{s}^{-1}\,\mathrm{m}^{-2}}
{\mathrm{C}\,\mathrm{mol}^{-1}}
\right]
=
\left[
\frac{\mathrm{mol}}
{\mathrm{m}^2\,\mathrm{s}}
\right].
\end{equation}

Note for it to work the c terms in the numerator on the left must be in $\text{mol}/\text{m}^3$ Therefore our concentration is volumetric in its derivation. And the reaction rate constant is really $1/\text{s}$ and it takes the volumetric concentration at the surface. No further brainstorming necessary even if I haven't explained the transition from volumetric to surface concentration perfectly.

In [54]:
#So the next state variable is essentially the same as the previous one.
#In case my assumptions are wrong I'll leave it as a separate variable for now.

v[variables["C_EC,s"]] = v[variables["c_EC"]]
v[variables["C_EC,s"]]

np.float64(4088121.735180558)

For the next variable, diffusivity, it needs to vary along with temperature, T. Therefore I'll initialize the latter first. 

To do this, I'm going to retroactively introduce a few new parameters idealizing the ambient temperature in the mine.

Newman et al. runs separate experiments setting the ambient temperature to 20 C and 40 C all the time, so I'll do the same.

In [55]:
parameters_names.append(["T_a", "ambient temperature", "K"])
prms["T_a"] = [294.15, 294.15]    #20 degrees Celsius in Kelvin

#Double-checks that parameters_names and prms have the same length and consistent keys.
assert len(parameters_names) == len(prms), "Mismatch between number of parameter names and prms dictionary"
for name in parameters_names:
    assert name[0] in prms, f"Parameter {name[0]} is missing in prms dictionary"
assert all(key in [name[0] for name in parameters_names] for key in prms), "Mismatch between prms dictionary keys and parameter names"

#Just in case it's more practical to work with, we're going to also create a numpy array of the parameters.
prms_arr = np.array([prms[name[0]] for name in parameters_names])
prms_arr.shape

v[variables["T"]] = prms["T_a"][battery_indx]

parameters_names.append(["T_ref", "reference temperature", "K"])
prms["T_ref"] = [298.15, 298.15]    #25 degrees Celsius in Kelvin

#Great we have what we need to initialize the Diffusivity state variable using the Arrhenius equation.
v[variables["D_EC"]] = prms["D_EC,Tref"][battery_indx] * np.exp(prms["E_a,D_EC"][battery_indx]/prms["R"][battery_indx] * (1/prms["T_ref"][battery_indx] - 1/v[variables["T"]]))

#I'll list both ref temperature value and real to see if the chance makes sense.
[v[variables["D_EC"]], prms["D_EC,Tref"][battery_indx]]

[np.float64(4.767864677288599e-19), np.float64(6.999999999999999e-19)]

In [56]:
#Next we go onto the effective diffusivity.
v[variables["D_EC,eff"]] = v[variables["D_EC"]] * prms["epsilon_sei"][battery_indx]**prms["n"][battery_indx]
v[variables["D_EC,eff"]]

np.float64(5.330634762968703e-21)

In [57]:
#Next we need the reaction rate constant from arrhenius.

v[variables["k_0,sei"]] = prms["k_0,sei,Tref"][battery_indx] * np.exp(prms["E_a,k_0,sei"][battery_indx]/prms["R"][battery_indx] * (1/prms["T_ref"][battery_indx] - 1/v[variables["T"]]))
v[variables["k_0,sei"]]

np.float64(3.4537587023712766e-15)

I'm beginning to realize that the breakdown of state variables has so far broken down into linear co-dependency and arrhenius. Fast iteration could almost be as simple as linear algebra. If only. We'll have to see what category of problem this ends up falling under and seeing what can be done to iterate it quickly. If necessary; it may be that 10000 cycles or whateer the expected number is is small enough that it doesn't need a whole bunch of computation time even at a sub-optimal algorithm.

However, if we end up optimizing then those 10000 might end up as part of an optimization variable. It remains to be seen how that should be formulated.

For now I'll keep initializing then when something breaks or is impossible I can approach it ad hoc.

In [58]:
#Next since we have the reaction rate constant, we can calculate the exchange current density.
v[variables["i_0,sei"]] = prms["F"][battery_indx] * v[variables["k_0,sei"]] * v[variables["C_EC,s"]]
v[variables["i_0,sei"]]

np.float64(0.0013623089600656975)

For electrode potential ... After some research I think I've found an okay middle ground between giving an arbitrary estimate for it and relyine on a complete implementation of the DFN (Doyle; Fuller; Newman) model like Newman et al. did using GT-Autolion to get these values (I'm very confident that's what they did). 

If push comes to shove and I can't recreate their data within some margin, I can resort to PyBaMM.

In Doyle; Fuller; Newman and Newman; Thomas-Alyea there is a spatially uniform idealization that I don't understand but can absolutely use.

\begin{equation}
\Phi_s-\Phi_e
\approx
U_{\mathrm{Gr}}(\theta,T)
+
\eta_{\mathrm{ct}}
+
i_{\mathrm{loc}}R_{\mathrm{film}}.
\end{equation}

- $\Phi_s$: Electric potential in the solid graphite phase, measured in volts (V).

- $\Phi_e$: Electric potential in the electrolyte adjacent to the graphite, measured in volts (V).

- $U_{\mathrm{Gr}}(\theta,T)$: Graphite equilibrium potential relative to the electrolyte, determined by the graphite lithiation fraction $\theta$ and temperature $T$, measured in volts (V).

- $\theta$: Graphite lithiation fraction (stoichiometric state of charge), dimensionless.

- $T$: Absolute temperature, measured in kelvin (K).

- $\eta_{\mathrm{ct}}$: Charge-transfer overpotential required to drive the graphite intercalation reaction at a finite rate, measured in volts (V).

- $i_{\mathrm{loc}}$: Local interfacial current density per unit graphite surface area, measured in $\mathrm{A\,m^{-2}}$.

- $R_{\mathrm{film}}$: Area-specific electrical resistance of the SEI film, measured in $\Omega\,\mathrm{m^2}$.

- $i_{\mathrm{loc}}R_{\mathrm{film}}$: Ohmic voltage drop across the SEI film, measured in volts (V).

In [59]:
#So we need new state variables, first for the lithiation fraction.

#At this point I don't want to scroll up and down so I'll check here if theta is already a term or no.
assert "theta" not in variables, "Theta is already defined in variables"

#Lovely, it's not... yet.
variables["theta"] = len(variables)
variables_names.append(["theta", "lithiation fraction", "-"])
v = np.append(v, 0)  # Initialize lithiation fraction to 0

assert len(variables_names) == len(v), "Mismatch between number of variable names and state variable vector length"

Love it. Now, Chen et al. (2020) listed material properties of a graphite anode and *even gave us the stoichiometry for the anode as a function of the SOC!*

Let's add those parameters and assume that the battery is dead.

In [60]:
#These values are gotten from the table 7 of Chen et al.
parameters_names.append(["theta_0", "lithiation fraction at SOC=0", "-"])
parameters_names.append(["theta_100", "lithiation fraction at SOC=100", "-"])
prms["theta_0"] = [0.0279, 0.0279]  
prms["theta_100"] = [0.9014, 0.9014] 

In [61]:
#To initialize this, we also need to initialize SOC. 
assert "SOC" not in variables, "SOC is already defined in variables"
variables["SOC"] = len(variables)
variables_names.append(["SOC", "state of charge", "-"])
v = np.append(v, 0)  # Initialize SOC to 0
assert len(variables_names) == len(v), "Mismatch between number of variable names and state variable vector length"

In [62]:
#I think I'll have it that the battery starts its digital twin life at 100% SOC.
v[variables["SOC"]] = 1.0
#Now we can initialize the lithiation fraction from the SOC.
v[variables["theta"]] = prms["theta_0"][0] + (prms["theta_100"][0] - prms["theta_0"][0]) * v[variables["SOC"]]
[v[variables["SOC"]], v[variables["theta"]]]

[np.float64(1.0), np.float64(0.9014)]

Now Chen et al. uses a really ... unexpected equation for OCV as U(x):

\begin{equation}
U_n(\theta)
=
1.9793\exp(-39.3631\theta)
+
0.2482
-
0.0909\tanh\!\left[29.8538(\theta-0.1234)\right]
-
0.04478\tanh\!\left[14.9159(\theta-0.2769)\right]
-
0.0205\tanh\!\left[30.4444(\theta-0.6103)\right].
\end{equation}

In [64]:
#Now ... I really don't want to call 4 tanh functions each iteration.
#So we will find U as a function of SOC and create a lookup table for it. This will be a 2D array with the first column being SOC and the second column being U.

def U_negative(theta: float | np.ndarray) -> float | np.ndarray:
    """Chen et al. negative-electrode OCV, in volts vs Li/Li+."""
    theta = np.asarray(theta)

    return (
        1.9793 * np.exp(-39.3631 * theta)
        + 0.2482
        - 0.0909 * np.tanh(29.8538 * (theta - 0.1234))
        - 0.04478 * np.tanh(14.9159 * (theta - 0.2769))
        - 0.0205 * np.tanh(30.4444 * (theta - 0.6103))
    )

SOC = np.linspace(0, 1, 1000)
U = U_negative(prms["theta_0"][battery_indx] + (prms["theta_100"][battery_indx] - prms["theta_0"][battery_indx]) * SOC)

U

array([1.0637404 , 1.04137659, 1.01976651, 0.9988845 , 0.97870578,
       0.95920638, 0.94036314, 0.92215368, 0.90455637, 0.88755028,
       0.8711152 , 0.85523158, 0.83988053, 0.82504379, 0.81070368,
       0.79684313, 0.78344563, 0.7704952 , 0.75797639, 0.74587428,
       0.7341744 , 0.72286278, 0.71192589, 0.70135065, 0.69112439,
       0.68123484, 0.67167016, 0.66241884, 0.65346977, 0.64481218,
       0.63643563, 0.62833002, 0.62048554, 0.61289271, 0.60554233,
       0.59842547, 0.59153347, 0.58485794, 0.57839073, 0.57212393,
       0.56604986, 0.56016104, 0.55445024, 0.5489104 , 0.54353468,
       0.53831639, 0.53324906, 0.52832636, 0.52354215, 0.51889043,
       0.51436536, 0.50996124, 0.50567252, 0.50149378, 0.49741972,
       0.49344517, 0.4895651 , 0.48577456, 0.48206874, 0.47844293,
       0.47489252, 0.47141302, 0.46800003, 0.46464925, 0.46135648,
       0.45811762, 0.45492866, 0.4517857 , 0.44868493, 0.44562263,
       0.44259521, 0.43959916, 0.43663108, 0.43368767, 0.43076

In [68]:
#Oh heck yeah, now Let's say SOC is 0.5124, then I can just interpolate.
SOC_value = 0.5124
U_value = np.interp(SOC_value, SOC, U)
[U_value, U[512]]
#It works!!

[np.float64(0.13324774182562146), np.float64(0.13324697080445116)]

Now for $\eta_\text{ct}$, the difference from electrochemical equilibrium necessary to sustain the desired current, we use a derivation from Butler-Volmer:

\begin{equation}
\eta_{\mathrm{ct}}
=
\frac{2RT}{F}
\operatorname{asinh}
\left(
\frac{i_{\mathrm{loc}}}{2i_0}
\right)
\end{equation}

- $\eta_{\mathrm{ct}}$: Charge-transfer overpotential, measured in volts (V). It is the additional voltage beyond equilibrium required to drive the graphite intercalation reaction at a finite rate.

- $R$: Universal gas constant, equal to $8.314\ \mathrm{J\,mol^{-1}\,K^{-1}}$.

- $T$: Absolute temperature, measured in kelvin (K).

- $F$: Faraday constant, equal to $96485\ \mathrm{C\,mol^{-1}}$.

- $i_{\mathrm{loc}}$: Local interfacial current density per unit graphite surface area, measured in $\mathrm{A\,m^{-2}}$.

- $i_0$: Exchange current density, measured in $\mathrm{A\,m^{-2}}$. It represents the intrinsic reaction rate of the graphite/electrolyte interface at electrochemical equilibrium.

Notice that we need $i_0$ here. Amazing (he said cynically) that it's a multivariate function as well:

\begin{equation}
i_0
=
F\,k_0\,
c_e^{\alpha}
\left(\theta c_{s,\max}\right)^{\alpha}
\left[(1-\theta)c_{s,\max}\right]^{\alpha}
\end{equation}

- $k_0$: Intercalation reaction-rate coefficient. Its units depend on the precise concentration convention used in the kinetic equation.

- $c_e$: Lithium-ion concentration in the electrolyte adjacent to the particle surface, measured in $\mathrm{mol\,m^{-3}}$.

- $\alpha$: Charge-transfer coefficient, dimensionless. For the symmetric Butler--Volmer form, $\alpha=0.5$.

- $\theta$: Negative-electrode lithiation fraction or stoichiometry, dimensionless.

- $c_{s,\max}$: Maximum lithium concentration that the negative-electrode active material can accommodate, measured in $\mathrm{mol\,m^{-3}}$.

- $\theta c_{s,\max}$: Current lithium concentration in the negative-electrode solid, measured in $\mathrm{mol\,m^{-3}}$.

- $(1-\theta)c_{s,\max}$: Concentration of remaining vacant lithium-storage sites in the negative-electrode solid, measured in $\mathrm{mol\,m^{-3}}$.

In [ ]:
#Chen et al gives us c_s,max so we can put that in immediately.
parameters_names.append(["c_s,max", "maximum lithium concentration in the solid phase", "mol/m3"])
prms["c_s,max"] = [29583, 29583]
#And they give us the intercalation reaction rate coefficient.
parameters_names.append(["k_0,Li", "intercalation reaction rate coefficient", "m/s"])

